# SQLite Interactive Demo
This notebook accompanies the SQLite presentation and covers database creation, raw SQL operations via the built-in `sqlite3` module, `Pandas` integration, and a basic `SQLAlchemy` ORM example using library-themed datasets.

## Working with `sqlite3`

### 0. Preliminaries

#### Categorization of SQL statements

| Category | Full Name | Purpose | Key Commands |
| :--- | :--- | :--- | :--- |
| **DDL** | Data Definition Language | Defines the structure or schema of the database. | `CREATE`, `ALTER`, `DROP`, `TRUNCATE` |
| **DML** | Data Manipulation Language | Modifies the actual data stored inside the tables. | `INSERT`, `UPDATE`, `DELETE` |
| **DQL** | Data Query Language | Retrieves or queries data from the database. | `SELECT` |
| **DCL** | Data Control Language | Manages permissions and access (absent in SQLite).| `GRANT`, `REVOKE` |
| **TCL** | Transaction Control Language | Manages database transactions for ACID compliance. | `COMMIT`, `ROLLBACK` |

#### The `sqlite3` Execution Lifecycle

Working with Python's native SQLite module follows a strict five-step sequence:

1. **Connect** (`sqlite3.connect()`): Opens a connection to the database file (creating it if it does not exist). This acts as the gateway to the database engine.
2. **Create Cursor** (`conn.cursor()`): Instantiates a cursor object. Think of this as the execution pointer used to send SQL commands and fetch results.
3. **Execute Command** (`cursor.execute()`): Sends the raw SQL string (DDL, DML, or DQL) to SQLite. For `SELECT` queries, results stay buffered in the cursor until retrieved via methods like `fetchall()`.
4. **Commit Changes** (`conn.commit()`): Persists write operations to disk. *Note: Read operations do not require a commit, but uncommitted write operations will be lost when the connection closes.*
5. **Close Connection** (`conn.close()`): Safely tears down the connection, releasing the file lock and freeing system resources.

### 1. Database Initialization & Schema Creation

In [4]:
import os
import sqlite3

# ensure data directory exists relative to the notebook location
os.makedirs("../data", exist_ok=True)
dbPath = "../data/library.db"

# establish connection
conn = sqlite3.connect(dbPath)

# obtain cursor handle
cursor = conn.cursor()

# execute multiple DDL statements to drop old tables and create fresh schema
cursor.executescript("""
DROP TABLE IF EXISTS loans;
DROP TABLE IF EXISTS patrons;
DROP TABLE IF EXISTS books;

CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    published_year INTEGER
);

CREATE TABLE patrons (
    patron_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    address TEXT NOT NULL
);

CREATE TABLE loans (
    loan_id INTEGER PRIMARY KEY AUTOINCREMENT,
    book_id INTEGER,
    patron_id INTEGER,
    loan_date TEXT,
    FOREIGN KEY (book_id) REFERENCES books (book_id),
    FOREIGN KEY (patron_id) REFERENCES patrons (patron_id)
);
""")

# note: executescript automatically commits changes to the database file (vs. execute() -> runs single statement at a time)
# close connection to safely release the file lock
conn.close()
print("Step 1 complete: Database initialized and schema created.")

Step 1 complete: Database initialized and schema created.


### 2. Sample Record Insertion

In [5]:
# reconnect to existing database file
dbPath = "../data/library.db"
conn = sqlite3.connect(dbPath)
cursor = conn.cursor()

# execute multiple DML statements to insert library records
cursor.executescript("""
INSERT INTO books (title, author, published_year) VALUES
('Meine Reisen auf dem Meeresgrund', 'Luigi Languste', 2017),
('Salate und ihre Zubereitung', 'Ludwig Landschildkröte', 2019),
('Schmackhafte Algen', 'Franziska Fisch', 2008),
('Zum Verbot aller Türen', 'Kerstin Katze', 1999);

INSERT INTO patrons (name, email, address) VALUES
('Testian Test', 'test@example.com', 'Testplatz 1'),
('Dirty Dieter', 'dieter@meta.com', 'SQL Island'),
('Hans Seas', 'hans@seas.at', 'Seven Seas'),
('Adelheid Meisenbaer', 'ameisen@baer.de', 'Tiergarten Schoenbrunn');

INSERT INTO loans (book_id, patron_id, loan_date) VALUES
(4, 1, '2026-08-01'),
(3, 2, '2026-08-10'),
(2, 4, '2026-08-14');
""")

# close connection after changes are implicitly committed by executescript
conn.close()
print("Step 2 complete: Inserted sample library data in DB.")

Step 2 complete: Inserted sample library data in DB.


### 3. Querying and Fetching Data from DB

In [6]:
import sqlite3

# reconnect to the database file
dbPath = "../data/library.db"
conn = sqlite3.connect(dbPath)
cursor = conn.cursor()

# execute a simple select query on a single table
cursor.execute("SELECT title, author, published_year FROM books WHERE published_year > 2010;")

# fetch all matching records from the cursor buffer
# returns list of tuples (= rows)
resultRows = cursor.fetchall()

# how does `rows` look
print(f"What does `rows` look like: {resultRows}\n")

# iterate over results
print("Books published after 2005:")
for row in resultRows:
    print(f"- {row[1]}: {row[0]} ({row[2]})")

# execute a more complex DQL query with inner joins across all three tables
query = """
SELECT b.title, b.author, p.name, l.loan_date
FROM loans l
JOIN books b ON l.book_id = b.book_id
JOIN patrons p ON l.patron_id = p.patron_id;
"""

cursor.execute(query)
advancedResultRows = cursor.fetchall()

# iterate over joined results
print("\nActive library loans:")
for row in advancedResultRows:
    print(f"- title: '{row[0]}' by {row[1]} | borrowed by: {row[2]} on {row[3]}")

# close connection
conn.close()

What does `rows` look like: [('Meine Reisen auf dem Meeresgrund', 'Luigi Languste', 2017), ('Salate und ihre Zubereitung', 'Ludwig Landschildkröte', 2019)]

Books published after 2005:
- Luigi Languste: Meine Reisen auf dem Meeresgrund (2017)
- Ludwig Landschildkröte: Salate und ihre Zubereitung (2019)

Active library loans:
- title: 'Zum Verbot aller Türen' by Kerstin Katze | borrowed by: Testian Test on 2026-08-01
- title: 'Schmackhafte Algen' by Franziska Fisch | borrowed by: Dirty Dieter on 2026-08-10
- title: 'Salate und ihre Zubereitung' by Ludwig Landschildkröte | borrowed by: Adelheid Meisenbaer on 2026-08-14


## `Pandas` Integration

### 0. Key Concepts

- **Bridging SQL and DataFrames:** Python's built-in `sqlite3` module handles raw execution, but **Pandas** bridges relational databases with data science workflows.
- **Reading Data (`pd.read_sql`):** Executes an SQL query and loads the result directly into a Pandas DataFrame in a single step, bypassing manual `cursor.fetchall()` loops.
- **Writing Data (`df.to_sql`):** Serializes a DataFrame straight into a new or existing SQLite database table without writing manual `INSERT` statements.

### 1. Reading Data

In [7]:
import sqlite3
import pandas as pd

# reconnect to the database file
dbPath = "../data/library.db"
conn = sqlite3.connect(dbPath)

# read an sql query directly into a pandas dataframe
booksDF = pd.read_sql("SELECT book_id, title, author, published_year FROM books;", conn)
patronsDF = pd.read_sql("SELECT patron_id, name, email, address FROM patrons;", conn)

# close connection after reading
conn.close()


In [8]:
booksDF

,book_id,title,author,published_year
0,1,Meine Reisen auf dem Meeresgrund,Luigi Languste,2017
1,2,Salate und ihre Zubereitung,Ludwig Landschildkröte,2019
2,3,Schmackhafte Algen,Franziska Fisch,2008
3,4,Zum Verbot aller Türen,Kerstin Katze,1999


In [9]:
patronsDF

,patron_id,name,email,address
0,1,Testian Test,test@example.com,Testplatz 1
1,2,Dirty Dieter,dieter@meta.com,SQL Island
2,3,Hans Seas,hans@seas.at,Seven Seas
3,4,Adelheid Meisenbaer,ameisen@baer.de,Tiergarten Schoenbrunn


### 2. Some Analyses

In [10]:
booksDF.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   book_id         4 non-null      int64
 1   title           4 non-null      str  
 2   author          4 non-null      str  
 3   published_year  4 non-null      int64
dtypes: int64(2), str(2)
memory usage: 260.0 bytes


In [11]:
booksDF[(booksDF["author"].str.startswith("L")) & (booksDF["title"].str.contains("Salat")) & (booksDF["published_year"] > 2017)]

,book_id,title,author,published_year
1,2,Salate und ihre Zubereitung,Ludwig Landschildkröte,2019


In [12]:
patronsDF.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   patron_id  4 non-null      int64
 1   name       4 non-null      str  
 2   email      4 non-null      str  
 3   address    4 non-null      str  
dtypes: int64(1), str(3)
memory usage: 260.0 bytes


In [13]:
def setDangerLevel(row):

    if row["address"] == "SQL Island" and row["name"] == "Dirty Dieter":
        return "DIETER!!!!"
    else:
        return "eh okay"

In [14]:
patronsDF["danger_level"] = patronsDF.apply(lambda row: setDangerLevel(row), axis=1)

In [15]:
patronsDF["danger_level"].value_counts()

danger_level
eh okay       3
DIETER!!!!    1
Name: count, dtype: int64

In [18]:
patronsDF

,patron_id,name,email,address,danger_level
0,1,Testian Test,test@example.com,Testplatz 1,eh okay
1,2,Dirty Dieter,dieter@meta.com,SQL Island,DIETER!!!!
2,3,Hans Seas,hans@seas.at,Seven Seas,eh okay
3,4,Adelheid Meisenbaer,ameisen@baer.de,Tiergarten Schoenbrunn,eh okay


In [16]:
patronsDF[patronsDF["danger_level"] == "DIETER!!!!"]

,patron_id,name,email,address,danger_level
1,2,Dirty Dieter,dieter@meta.com,SQL Island,DIETER!!!!


### 3. Writing Data

In [27]:
# let's add some raccoons

import pandas as pd
import sqlite3

dbPath = "../data/library.db"
conn = sqlite3.connect(dbPath)

raccoonDF = pd.DataFrame.from_dict(
    {
        "raccoon_id": [1, 2, 3],
        "raccoon_name": ["Wilfried", "Walpurga", "Wolfgang"],
        "workplace": ["Buchausgabe", "Hauptabteilungsleitung", "ITS"]
    }
)

# set index
raccoonDF= raccoonDF.set_index("raccoon_id")

# write to db
raccoonDF.to_sql("raccoons", conn, if_exists="replace", index=True, index_label="raccoon_id")

conn.close()

print("Added important table to database.")

Added important table to database.


In [28]:
# let's warn people about Dirty Dieter!

import sqlite3
import pandas as pd

dbPath = "../data/library.db"
conn = sqlite3.connect(dbPath)

# write the modified dataframe back to sqlite (replacing the table)
patronsDF.to_sql("patrons_warnings", conn, if_exists="replace", index=False)

# close connection
conn.close()

print("Dirty Dieter is flagged. People are safe now!")

Dirty Dieter is flagged. People are safe now!


## Object-Relational Mapping (ORM) with `SQLAlchemy`

### 0. Key Concepts

- **Models**: Python classes that inherit from a declarative base, defining table schemas and constraints natively in code.
- **Engine & Session**: The engine manages the actual database connection, while the session handles transactions, queries, and persistence.
- **Object-Oriented Operations**: Records are manipulated as standard Python objects rather than raw tuples or dictionaries.

### 1. Create Models

In [20]:
from sqlalchemy import Integer, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

# base class
class Base(DeclarativeBase):
    pass

# class for table
class Book(Base):
    __tablename__ = "books"
    
    bookId: Mapped[int] = mapped_column("book_id", Integer, primary_key=True, autoincrement=True)
    title: Mapped[str] = mapped_column("title", String(100), nullable=False)
    author: Mapped[str] = mapped_column("author", String(100))
    publishedYear: Mapped[int] = mapped_column("published_year", Integer)


### 2. Use Engine & Session for Sample Record Creation

In [21]:
from sqlalchemy import create_engine

# create database engine
# engine serves as central control hub for database, 
# managing connection pooling, communicating with the file path, 
# and translating Python/SQLAlchemy commands into the correct SQL dialect
engine = create_engine("sqlite:///../data/library_orm.db")

# create table in datebase
Base.metadata.create_all(engine)

print("Database and tables created via SQLAlchemy models.")

Database and tables created via SQLAlchemy models.


In [22]:
# create sample data using the Book class

book1 = Book(
    title="Meine motorisierten Reisen auf dem Meeresgrund",
    author="Luigi Languste II.",
    publishedYear=2012
)

book2 = Book(
    title="Rache für meinen Vater",
    author="Luigi Languste II.",
    publishedYear=2015
)

In [25]:
# write book records to database using Session

from sqlalchemy.orm import Session

with Session(engine) as session:
  session.add(book1)
  session.add(book2)
  # could also use: 
  # session.add_all([book1, book2])
  session.commit()

print(f"Added to DB: {str(book1)}, {str(book2)}")

Added to DB: <__main__.Book object at 0x0000019BEED67A10>, <__main__.Book object at 0x0000019BEF158B90>


In [15]:
# read data via orm queries

from sqlalchemy.orm import Session

with Session(engine) as session:

    allBooks = session.query(Book).all()

    #print(f"{allBooks}\n")

    for book in allBooks:
        print(f"ID: {book.bookId} | Title: {book.title} by {book.author}")


ID: 1 | Title: Meine motorisierten Reisen auf dem Meeresgrund by Luigi Languste II.
ID: 2 | Title: Rache für meinen Vater by Luigi Languste II.


In [26]:
# use this in case of wanting to delete db
engine.dispose()

In [ ]:
# this is how it could also look

from datetime import datetime
from typing import List, Optional

from sqlalchemy import (
    Column,
    DateTime,
    Float,
    ForeignKey,
    Integer,
    String,
    Table,
    Text,
    func,
)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


# Base Class
class Base(DeclarativeBase):
    pass


# Junction Table: Records <-> URLs (Many-to-Many)
recordUrls = Table(
    "record_urls",
    Base.metadata,
    Column("record_id", Integer, ForeignKey("records.record_id"), primary_key=True),
    Column("url_id", Integer, ForeignKey("urls.url_id"), primary_key=True),
    Column("marc_tag", String(10), nullable=True),  # e.g., '856$u'
)

# Junction Table: Records <-> Sets (Many-to-Many)
recordSets = Table(
    "record_sets",
    Base.metadata,
    Column("record_id", Integer, ForeignKey("records.record_id"), primary_key=True),
    Column("set_id", Integer, ForeignKey("sets.set_id"), primary_key=True),
)


# Sets Table
class Set(Base):
    __tablename__ = "sets"

    id: Mapped[int] = mapped_column("set_id", Integer, primary_key=True, autoincrement=True)
    setSpec: Mapped[str] = mapped_column("set_spec", String(100), unique=True, nullable=False, index=True)
    name: Mapped[str] = mapped_column("name", String(255), nullable=False)
    sourceType: Mapped[str] = mapped_column("source_type", String(50), nullable=False)  # 'Alma API' or 'OAI-PMH'
    createdAt: Mapped[datetime] = mapped_column("created_at", DateTime(timezone=True), server_default=func.now())

    records: Mapped[List["Record"]] = relationship(
        "Record", secondary=recordSets, back_populates="sets"
    )


# Records Table
class Record(Base):
    __tablename__ = "records"

    id: Mapped[int] = mapped_column("record_id", Integer, primary_key=True, autoincrement=True)
    mmsId: Mapped[str] = mapped_column("mms_id", String(50), unique=True, nullable=False, index=True)
    marcTitlesJson: Mapped[Optional[str]] = mapped_column("marc_titles_json", Text, nullable=True)
    createdAt: Mapped[datetime] = mapped_column("created_at", DateTime(timezone=True), server_default=func.now())

    sets: Mapped[List["Set"]] = relationship(
        "Set", secondary=recordSets, back_populates="records"
    )
    urls: Mapped[List["URL"]] = relationship(
        "URL", secondary=recordUrls, back_populates="records"
    )


# URLs Table
class URL(Base):
    __tablename__ = "urls"

    id: Mapped[int] = mapped_column("url_id", Integer, primary_key=True, autoincrement=True)
    urlString: Mapped[str] = mapped_column("url_string", String(2048), unique=True, nullable=False, index=True)
    domain: Mapped[Optional[str]] = mapped_column("domain", String(255), index=True)

    records: Mapped[List["Record"]] = relationship(
        "Record", secondary=recordUrls, back_populates="urls"
    )
    checks: Mapped[List["URLCheck"]] = relationship("URLCheck", back_populates="url")


# URL Checks Table
class URLCheck(Base):
    __tablename__ = "url_checks"

    id: Mapped[int] = mapped_column("check_id", Integer, primary_key=True, autoincrement=True)
    urlId: Mapped[int] = mapped_column("url_id", Integer, ForeignKey("urls.url_id"), nullable=False)

    statusCode: Mapped[Optional[int]] = mapped_column("status_code", Integer)
    responseTimeMs: Mapped[Optional[float]] = mapped_column("response_time_ms", Float)
    errorMessage: Mapped[Optional[str]] = mapped_column("error_message", Text)

    htmlTitle: Mapped[Optional[str]] = mapped_column("html_title", Text)
    titleMatchScore: Mapped[Optional[float]] = mapped_column("title_match_score", Float)
    titleMatchPass: Mapped[Optional[int]] = mapped_column("title_match_pass", Integer)

    createdAt: Mapped[datetime] = mapped_column("created_at", DateTime(timezone=True), server_default=func.now())

    url: Mapped["URL"] = relationship("URL", back_populates="checks")